In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

train = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/train.csv')
test = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/test.csv')
sample = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/Sample.csv')


In [ ]:
- SMOTE
- CHECKPOINTS
- NAIVE BAYES/ Logistic regression add probs to features (BOOSTING)

TO DO FOR TOMORROW:
- Train final model on entire dataset.
- Increase percentile to 20%
- Increase estimators.
- Do better tuning.

In [2]:
import gc
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer


In [3]:

train['is_train'] = 1
test['is_train'] = 0

test_temp = test.copy()
test_temp['label'] = np.nan 

all_data = pd.concat([train, test_temp], axis=0, ignore_index=True)

thread_stats = all_data.groupby('post_id').agg({
    'comment': 'count',              # How long is the thread? (Long threads = arguments)
    'upvote': ['mean', 'std', 'sum'], # Is the thread popular?
    'downvote': ['mean', 'std', 'sum'] # Is the thread hated/controversial?
})

thread_stats.columns = [
    'thread_vol', 
    'thread_up_mean', 'thread_up_std', 'thread_up_sum', 
    'thread_down_mean', 'thread_down_std', 'thread_down_sum'
]
thread_stats = thread_stats.reset_index()

thread_stats['thread_engagement'] = thread_stats['thread_up_sum'] + thread_stats['thread_down_sum']
thread_stats['thread_negativity_ratio'] = thread_stats['thread_down_sum'] / (thread_stats['thread_up_sum'] + 1)

train = train.merge(thread_stats, on='post_id', how='left')
test = test.merge(thread_stats, on='post_id', how='left')

fill_zero_cols = ['thread_up_std', 'thread_down_std']
train[fill_zero_cols] = train[fill_zero_cols].fillna(0)
test[fill_zero_cols] = test[fill_zero_cols].fillna(0)

print("Thread features added successfully.")

Thread features added successfully.


In [5]:
import re

def add_term_indicator_columns(
    df,
    terms,
    prefix,
    case_insensitive=True
):
    df = df.copy()

    flags = re.IGNORECASE if case_insensitive else 0

    for term in terms:
        col_name = f"{prefix}_{term}"
        pattern = re.escape(term)
        df[col_name] = df["comment"].str.contains(
            pattern,
            regex=True,
            flags=flags,
            na=False
        )

    return df


In [6]:
# === Feature Engineering and Preprocessing ===
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import FeatureUnion


thread_features = [
    'thread_vol', 'thread_up_mean', 'thread_up_std', 
    'thread_down_mean', 'thread_down_std', 
    'thread_engagement', 'thread_negativity_ratio'
]

# Update X_full to include these new columns
# Note: We added 'post_id' to the list just in case, but we won't use it in the model directly
cols_to_use = [
    'emoticon_1', 'emoticon_2', 'emoticon_3', 
    'upvote', 'downvote', "if_1", "if_2", 
    "disability", "race", "religion", "gender", 
    'comment'
] + thread_features 

X_full = train[cols_to_use].copy()
y = train['label']

X_full["text_len"] = X_full["comment"].str.len()
X_full["word_count"] = X_full["comment"].str.split().str.len()

X_full["question_marks"] = X_full["comment"].str.count(r"\?")
X_full["exclamations"] = X_full["comment"].str.count(r"!")

X_full[["text_len", "word_count"]]  = X_full[["text_len", "word_count"]].fillna(0)
X_full[["question_marks", "exclamations"]] = X_full[["question_marks", "exclamations"]].fillna(False)

X_full = X_full.fillna('Null')

import re
# "I will kiiillll youuuu" -> "I will kill you"
X_full['comment'] = X_full['comment'].apply(lambda x: re.sub(r'(.)\1{2,}', r'\1', str(x)))

violent_terms = [
    'kill', 'die', 'death', 'shoot', 'fire', 'killing', 'dead', 'hang', 'hit',
    'burn', 'murder', 'rid', 'shot', 'punch', 'shooting', 'guns', 'suicide',
    'bomb', 'gun', 'fired', 'kick', 'knife', 'killed', 'torture', 'bullet',
    'slap', 'destroy', 'stab', 'kill', 'killed', 'killer', 'murder', 'murderer',
    'suicide', 'nuke', 'pulverize', 'holocaust', 'dead', 'death', 'die',
    'gun', 'fire', 'shoot', 'bullet', 'missile', 'blow up', 'poison',
    'dumping body', 'strike fear', 'jail', 'prison', 'lock her up', 'prosecute',
    'death penalty', 'terrorist', 'meth dealer', 'war zone', 'sanctuary status'
]


X_full = add_term_indicator_columns(X_full,terms=violent_terms,prefix="v")

moral_terms = ["deserve", "should", "solution", "need to", "have to", "all of them"]
X_full = add_term_indicator_columns(X_full,terms=moral_terms,prefix="m")

group_terms = [
    'muslim', 'jew', 'jewish', 'christian', 'catholic', 'women', 'men',
    'female', 'male', 'black', 'white', 'asian', 'immigrant', 'refugee',
    'gay', 'lesbian', 'native', 'indian', 'muslim', 'catholic', 'jewish',
    'christian', 'buddhist', 'infidel', 'haram', 'devout', 'religious',
    'white', 'black', 'native american', 'ethnic', 'liberal', 'left winger',
    'progressive', 'theo-con', 'trump', 'blm', 'ku klux klan', 'neo-nazi',
    'white supremacist', 'woman', 'female', 'male', 'gay', 'homosexual',
    'refugee', 'asylum seeker'
              ]

X_full = add_term_indicator_columns(X_full,terms=group_terms,prefix="g")

personal_attack = [
    "lunatic", "looney", "mindless drivel", "ignorant", "ignorance",
    "stupid", "foolish", "gullible", "hallucination", "mental condition",
    "greedy", "criminal", "ethically challenged", "corruption", "rot",
    "cultist", "sheep", "loudmouth", "vampire", "traitor", 
    "troll", "arrogant", "liar", "lying", "weak", "victim complex",
    "mom's basement", "strawman", "hypocrite"
                  ]

X_full = add_term_indicator_columns(X_full,terms=personal_attack,prefix="p")

# Create en vioxplicit count features for the most commolent roots
violent_roots = ['kill', 'shoot', 'shot', 'dead', 'death', 'die', 'hang', 'gun', 'murder']

for word in violent_roots:
    # Use regex \b to match exact word boundaries, avoiding "skill" when looking for "kill"
    X_full[f'count_{word}'] = X_full['comment'].str.count(rf'(?i)\b{word}\b')

# Create a total "Violence Density" score
X_full['total_violent_words'] = X_full[[f'count_{w}' for w in violent_roots]].sum(axis=1)
X_full['violence_density'] = X_full['total_violent_words'] / (X_full['comment'].str.len() + 1)

# Count direct targets
X_full['count_you'] = X_full['comment'].str.count(r'(?i)\b(you|your|ur)\b')
X_full['count_they'] = X_full['comment'].str.count(r'(?i)\b(they|them|their)\b')

# Create an interaction feature: Do they use a violent word AND a targeting pronoun?
X_full['targeted_violence_flag'] = np.where(
    (X_full['total_violent_words'] > 0) & ((X_full['count_you'] > 0) | (X_full['count_they'] > 0)), 
    1, 0
)



X_full["caps_ratio"] = X_full["comment"].apply(lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)), 1))

test = test[cols_to_use].copy()
# y = train['label']

test["text_len"] = test["comment"].str.len()
test["word_count"] = test["comment"].str.split().str.len()

test["question_marks"] = test["comment"].str.count(r"\?")
test["exclamations"] = test["comment"].str.count(r"!")

test[["text_len", "word_count"]]  = test[["text_len", "word_count"]].fillna(0)
test[["question_marks", "exclamations",]] = test[["question_marks", "exclamations"]].fillna(False)
test["caps_ratio"] = test["comment"].apply(lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)), 1))


test = test.fillna('Null')
test = add_term_indicator_columns(test,terms=violent_terms,prefix="v")
test = add_term_indicator_columns(test,terms=moral_terms,prefix="m")
test = add_term_indicator_columns(test,terms=group_terms,prefix="g")
test = add_term_indicator_columns(test,terms=personal_attack,prefix="p")




base_num_features = [
    'emoticon_1', 'emoticon_2', 'emoticon_3', 'upvote', 'downvote', "if_1", "if_2", 
    "disability", "text_len", "word_count", "question_marks", "exclamations", "caps_ratio"
    ] + thread_features

term_feature_cols = [
    c for c in X_full.columns
    if c.startswith(("v_", "m_", "g_", "p_"))
]

num_features = base_num_features + term_feature_cols
# text_feature = "comment"
cat_features = ["race", "religion", "gender"]


feature_encoder = ColumnTransformer(
    transformers=[
        # ("text_vec", TfidfVectorizer(max_features=20000,ngram_range=(1, 2)), text_feature),
        ("num_scaler", StandardScaler(), num_features), 
        ("cats", OneHotEncoder(handle_unknown="ignore"), cat_features)
    ],
)



In [7]:
X_full.columns

Index(['emoticon_1', 'emoticon_2', 'emoticon_3', 'upvote', 'downvote', 'if_1',
       'if_2', 'disability', 'race', 'religion',
       ...
       'count_die', 'count_hang', 'count_gun', 'count_murder',
       'total_violent_words', 'violence_density', 'count_you', 'count_they',
       'targeted_violence_flag', 'caps_ratio'],
      dtype='object', length=157)

In [6]:
# ------------------------------------------------------------------------------
print("Vectorizing Text (Max Parameters)...")


# Pipeline A: Word N-Grams (Context)
# Captures "kill you" vs "kill time"
word_vec = TfidfVectorizer(
    max_features=50000,     
    ngram_range=(1, 3),     
    sublinear_tf=True,      
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'\w{1,}',
    min_df=3,                     # Added: Ignores typos that appear < 3 times
    max_df=0.9                    # Added: Ignores words in >90% of docs
)

# Pipeline B: Character N-Grams (Leave this mostly alone!)
# Do NOT apply stop_words here, let it find character patterns natively
char_vec = TfidfVectorizer(
    max_features=50000,     
    ngram_range=(3, 5),     
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='char_wb',
    min_df=5,                     # Added: Drop ultra-rare character hashes
    max_df=0.9                    # Added: Drop ultra-common character hashes
)

# Fit on ALL text (Train + Test) to ensure shared vocabulary
# (This is allowed and standard practice in Kaggle)
all_text = pd.concat([X_full['comment'], test['comment']])
word_vec.fit(all_text)
char_vec.fit(all_text)

# Transform Train and Test
train_word = word_vec.transform(X_full['comment'])
train_char = char_vec.transform(X_full['comment'])
test_word = word_vec.transform(test['comment'])
test_char = char_vec.transform(test['comment'])

# Combine Text Features
X_train_text = hstack([train_word, train_char])
X_test_text = hstack([test_word, test_char])

print(f"Text Feature Shape: {X_train_text.shape}")



Vectorizing Text (Max Parameters)...
Text Feature Shape: (198000, 100000)


In [7]:
from sklearn.feature_selection import SelectPercentile, chi2
import numpy as np

print('starting selector ')

# 1. Initialize the selector
# We keep only the top 20% of the 45,000 features (approx 9,000 features)
# This usually retains ~98% of the model's predictive power.
selector = SelectPercentile(chi2, percentile=25)

print(f"Original shape: {X_train_text.shape}")

# 2. Fit on training data and transform both
X_train_text_reduced = selector.fit_transform(X_train_text, y)
X_test_text_reduced = selector.transform(X_test_text)

print(f"Reduced shape: {X_train_text_reduced.shape}")

starting selector 
Original shape: (198000, 100000)
Reduced shape: (198000, 25000)


In [8]:
X_train_meta = feature_encoder.fit_transform(X_full)
X_test_meta = feature_encoder.transform(test)

# Combine EVERYTHING for LightGBM (Sparse + Dense)
X_full_train = hstack([X_train_text_reduced, X_train_meta]).tocsr()
X_full_test = hstack([X_test_text_reduced, X_test_meta]).tocsr()




In [9]:
import lightgbm as lgb
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
import time


X_train, X_valid, y_train, y_valid = train_test_split(
    X_full_train, y, test_size=0.2, random_state=0, stratify=y
)


modelA = lgb.LGBMClassifier(
        n_estimators=3000,
        learning_rate=0.05,
        num_leaves=96,
        max_depth=15,
        min_child_samples=30,
        subsample=0.8,
        colsample_bytree=0.1,
        reg_alpha=0.5,
        reg_lambda=1.5,
        n_jobs=-1,
        random_state=0,
        verbosity=-1
        )



In [10]:
from lightgbm import log_evaluation, early_stopping


In [ ]:
import lightgbm as lgb
from lightgbm import log_evaluation, early_stopping
import time

# 1. Setup Callbacks
# log_evaluation(100) will print progress every 100 iterations
# early_stopping(100) stops training if the validation score doesn't improve for 100 rounds
callbacks = [
    log_evaluation(period=100), 
    # early_stopping(stopping_rounds=100)
]

start_time = time.time()

# 2. Fit with progress trackinga
modelA.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],  # Tracking performance on validation set
    eval_metric='multi_logloss',    # Standard for multiclass progress tracking
    callbacks=callbacks
)

end_time = time.time()
training_time = end_time - start_time
print(f"\n--- Training Finished ---")
print(f"Total Training Time: {training_time/60:.2f} minutes.")


best_iter = modelA.best_iteration_
print(f"\nBest Iteration found: {best_iter}")


# 3. Final Evaluation
preds = modelA.predict(X_valid)
print(classification_report(y_valid, preds))

11:40 start


In [14]:
from scipy.optimize import minimize
from sklearn.metrics import f1_score, classification_report
import numpy as np

In [ ]:
from scipy.optimize import minimize
from sklearn.metrics import f1_score, classification_report
import numpy as np

# ==========================================
# PHASE 4 (HYBRID): AGGRESSIVE OPTIMIZATION
# ==========================================

print("Generating validation probabilities...")
val_probs = modelA.predict_proba(X_valid) 

In [ ]:
# 1. Define the Loss Function (Negative F1)
def f1_loss(weights):
    # Clip weights to be non-negative
    w = np.maximum(weights, 0) 
    weighted_probs = val_probs * w
    final_preds = np.argmax(weighted_probs, axis=1)
    return -f1_score(y_valid, final_preds, average='macro')

# 2. Define a list of diverse starting points (The "Candidates")
# We focus on variations around your successful manual logic
init_candidates = [
    # Baseline
    [1.0, 1.0, 1.0, 1.0],
    
    # Your manual "Sweet Spot" approximations
    [1.0, 1.0, 1.0, 4.0],  # ~0.25 threshold
    [1.0, 1.0, 1.0, 5.0],  # ~0.20 threshold
    [1.0, 1.0, 1.0, 6.0],  # ~0.16 threshold
    
    # High Aggression on Hate (1) and Violence (3)
    [1.0, 1.5, 1.2, 5.0],  
    [0.9, 1.3, 1.1, 4.5],
    
    # Suppress Civil (0) to force hard calls
    [0.8, 1.2, 1.2, 4.0],
    
    # "The Kitchen Sink" (High variance)
    [0.5, 2.0, 2.0, 8.0],

    #MANUAL TRY
    [1.0, 2.0, 1.2, 5.5],
    [1, 1.5, 1.2, 5],
    [1.0, 2.0, 1.5, 5],
    [1.0, 1.2, 1.0, 4],
    [0.8, 1.3, 1.1, 4.5],
]

best_global_score = -1
best_global_weights = None
best_start_point = None

# 3. The Loop (Try all starting points)
for i, init_w in enumerate(init_candidates):
    print(f"\n[Attempt {i+1}/{len(init_candidates)}] Starting from: {init_w}")
    
    # Run Optimizer
    opt_result = minimize(
        f1_loss, 
        init_w, 
        method='Powell', 
        tol=1e-5,
        options={'maxiter': 500}
    )
    
    current_score = -opt_result.fun
    current_weights = np.maximum(opt_result.x, 0) # Clip negative results
    
    print(f"  -> Result F1: {current_score:.5f}")
    
    # Update Winner
    if current_score > best_global_score:
        best_global_score = current_score
        best_global_weights = current_weights
        best_start_point = init_w
        print(f"  -> ⭐ NEW BEST FOUND!")

# 4. Compare with Pure Manual Threshold (Safety Check)
manual_preds = np.argmax(val_probs, axis=1)
manual_preds[val_probs[:, 3] > 0.2] = 3
manual_f1 = f1_score(y_valid, manual_preds, average='macro')

print("\n" + "="*40)
print(f"FINAL RESULTS")
print("="*40)
print(f"Manual Threshold (0.2) F1 : {manual_f1:.5f}")
print(f"Optimizer Best F1         : {best_global_score:.5f}")

if best_global_score > manual_f1:
    print("✅ Optimization Winner! Using optimized weights.")
    final_weights = best_global_weights
else:
    print("⚠️ Manual Threshold logic is simpler and better. Using Manual equivalent.")
    # Approximate the 0.2 threshold behavior with weights
    final_weights = np.array([1.0, 1.0, 1.0, 5.0]) 

print(f"Final Weights Chosen: {np.round(final_weights, 3)}")

Running Hybrid Optimization (Starting with Aggressive Class 3)...
------------------------------
Optimized Macro F1: 0.82774
Best Class Weights: [1.045 1.638 1.14  3.783]
------------------------------
Your Manual Threshold F1: 0.82518
✅ Optimization Found a Better Strategy!

In [11]:
model_final = lgb.LGBMClassifier(
        n_estimators=3000,
        learning_rate=0.05,
        num_leaves=96,
        max_depth=15,
        min_child_samples=30,
        subsample=0.8,
        colsample_bytree=0.1,
        reg_alpha=0.5,
        reg_lambda=1.5,
        n_jobs=-1,
        random_state=0,
        verbosity=-1
        )

# Fit on X_full_train (Combined Train + Val)
model_final.fit(X_full_train, y)

start_time = time.time()

# 2. Fit with progress tracking
model_final.fit(
    X_full_train, y,
    # eval_set=[(X_valid, y_valid)],  # Tracking performance on validation set
    # eval_metric='multi_logloss',    # Standard for multiclass progress tracking
    callbacks=callbacks
)

end_time = time.time()
training_time = end_time - start_time
print(f"\n--- Training Finished ---")
print(f"Total Training Time: {training_time/60:.2f} minutes.")


NameError: name 'callbacks' is not defined

Model training started at 6:04pm

In [12]:
model_final

LGBMClassifier(colsample_bytree=0.1, learning_rate=0.05, max_depth=15,
               min_child_samples=30, n_estimators=3000, n_jobs=-1,
               num_leaves=96, random_state=0, reg_alpha=0.5, reg_lambda=1.5,
               subsample=0.8, verbosity=-1)

In [15]:
print("Generating validation probabilities...")
val_probs = model_final.predict_proba(X_valid) 

# 1. Define the Loss Function (Negative F1)
def f1_loss(weights):
    # Clip weights to be non-negative
    w = np.maximum(weights, 0) 
    weighted_probs = val_probs * w
    final_preds = np.argmax(weighted_probs, axis=1)
    return -f1_score(y_valid, final_preds, average='macro')

# 2. Define a list of diverse starting points (The "Candidates")
# We focus on variations around your successful manual logic
init_candidates = [
    # Baseline
    # [1.0, 1.0, 1.0, 1.0],
    
    # # Your manual "Sweet Spot" approximations
    # [1.0, 1.0, 1.0, 4.0],  # ~0.25 threshold
    # [1.0, 1.0, 1.0, 5.0],  # ~0.20 threshold
    # [1.0, 1.0, 1.0, 6.0],  # ~0.16 threshold
    
    # # High Aggression on Hate (1) and Violence (3)
    # [1.0, 1.5, 1.2, 5.0],  
    # [0.9, 1.3, 1.1, 4.5],
    
    # # Suppress Civil (0) to force hard calls
    # [0.8, 1.2, 1.2, 4.0],
    
    # # "The Kitchen Sink" (High variance)
    # [0.5, 2.0, 2.0, 8.0],

    #MANUAL TRY
    # [1.0, 2.0, 1.2, 5.5],
    [1, 1.5, 1.2, 5],
    # [1.0, 2.0, 1.5, 5],
    # [1.0, 1.2, 1.0, 4],
    # [0.8, 1.3, 1.1, 4.5],
]

best_global_score = -1
best_global_weights = None
best_start_point = None

# 3. The Loop (Try all starting points)
for i, init_w in enumerate(init_candidates):
    print(f"\n[Attempt {i+1}/{len(init_candidates)}] Starting from: {init_w}")
    
    # Run Optimizer
    opt_result = minimize(
        f1_loss, 
        init_w, 
        method='Powell', 
        tol=1e-5,
        options={'maxiter': 500}
    )
    
    current_score = -opt_result.fun
    current_weights = np.maximum(opt_result.x, 0) # Clip negative results
    
    print(f"  -> Result F1: {current_score:.5f}")
    
    # Update Winner
    if current_score > best_global_score:
        best_global_score = current_score
        best_global_weights = current_weights
        best_start_point = init_w
        print(f"  -> ⭐ NEW BEST FOUND!")

# 4. Compare with Pure Manual Threshold (Safety Check)
manual_preds = np.argmax(val_probs, axis=1)
manual_preds[val_probs[:, 3] > 0.2] = 3
manual_f1 = f1_score(y_valid, manual_preds, average='macro')

print("\n" + "="*40)
print(f"FINAL RESULTS")
print("="*40)
print(f"Manual Threshold (0.2) F1 : {manual_f1:.5f}")
print(f"Optimizer Best F1         : {best_global_score:.5f}")

if best_global_score > manual_f1:
    print("✅ Optimization Winner! Using optimized weights.")
    final_weights = best_global_weights
else:
    print("⚠️ Manual Threshold logic is simpler and better. Using Manual equivalent.")
    # Approximate the 0.2 threshold behavior with weights
    final_weights = np.array([1.0, 1.0, 1.0, 5.0]) 

print(f"Final Weights Chosen: {np.round(final_weights, 3)}")

Generating validation probabilities...

[Attempt 1/1] Starting from: [1, 1.5, 1.2, 5]
  -> Result F1: 0.99997
  -> ⭐ NEW BEST FOUND!

FINAL RESULTS
Manual Threshold (0.2) F1 : 0.99983
Optimizer Best F1         : 0.99997
✅ Optimization Winner! Using optimized weights.
Final Weights Chosen: [2.002 3.885 2.391 5.625]


In [17]:
final_weights = [2.967, 4.65, 2.034, 14.567]

[1.248 2.273 1.235 4.461]

In [19]:
test_probs = model_final.predict_proba(X_full_test)
weighted_test_probs = test_probs * final_weights
final_test_preds = np.argmax(weighted_test_probs, axis=1)

# Load sample submission to get correct format
sample_submission = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/Sample.csv')

# Create submission dataframe
submission = pd.DataFrame({
    sample_submission.columns[0]: sample_submission.iloc[:, 0],  # ID column (first column)
    'label': final_test_preds  # Predictions
})

# Verify submission format
print("\nSubmission Preview:")
print(submission.head())
print(f"\nSubmission shape: {submission.shape}")
print(f"Prediction distribution:\n{submission['label'].value_counts().sort_index()}")

# Save submission file
submission.to_csv('submission_maxed_11.csv', index=False)
print("\n✓ Submission file saved as 'submission.csv'")

# Verify no missing values
assert submission.isnull().sum().sum() == 0, "WARNING: Submission has missing values!"
print("✓ No missing values in submission")



Submission Preview:
   ID  label
0   1      2
1   2      2
2   3      0
3   4      0
4   5      2

Submission shape: (102000, 2)
Prediction distribution:
label
0    57149
1     8630
2    33216
3     3005
Name: count, dtype: int64

✓ Submission file saved as 'submission.csv'
✓ No missing values in submission


In [ ]:
import lightgbm as lgb
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
import time

# ------------------------------------------------------------------------------
# 2. THE 5-FOLD CROSS-VALIDATION LOOP
# ------------------------------------------------------------------------------
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

# Arrays to store predictions
oof_preds_lgb = np.zeros((X_full_train.shape[0], 4))
test_preds_lgb = np.zeros((X_full_test.shape[0], 4))

oof_preds_lr = np.zeros((X_full_train.shape[0], 4))
test_preds_lr = np.zeros((X_full_test.shape[0], 4))

print(f"\nStarting {n_folds}-Fold Cross-Validation...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_full_train, y)):
    
    # Split Data
    X_tr, X_val = X_full_train[train_idx], X_full_train[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    
    # --- MODEL 1: LIGHTGBM (The Heavy Hitter) ---
    # Maxed out parameters as requested
    lgb_clf = lgb.LGBMClassifier(
        n_estimators=200,          # Massive number of trees
        learning_rate=0.01,         # Slow learning for better convergence
        num_leaves=128,             # Higher complexity
        max_depth=15,               # Deeper trees
        min_child_samples=20,
        subsample=0.7,
        colsample_bytree=0.6,       # Feature sampling
        reg_alpha=0.5,
        reg_lambda=1.5,             # Regularization to prevent overfitting
        # class_weight='balanced',    # Critical for Class 3
        n_jobs=-1,
        random_state=42,
        verbosity=-1
    )
    
    # Early Stopping is vital when n_estimators is high
    # We use a callback for early stopping
    callbacks = [lgb.early_stopping(stopping_rounds=100, verbose=False)]
    
    lgb_clf.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric='multi_logloss',
        callbacks=callbacks
    )
    
    # Store Predictions
    oof_preds_lgb[val_idx] = lgb_clf.predict_proba(X_val)
    test_preds_lgb += lgb_clf.predict_proba(X_full_test) / n_folds
    
    # --- MODEL 2: LOGISTIC REGRESSION (The Diversity Hire) ---
    # Linear models capture simple text signals that trees miss
    lr_clf = LogisticRegression(
        C=2.0,                  # Stronger regularization
        solver='saga',          # Fast for large sparse data
        class_weight='balanced',
        max_iter=300,
        n_jobs=-1
    )
    
    lr_clf.fit(X_tr, y_tr)
    
    oof_preds_lr[val_idx] = lr_clf.predict_proba(X_val)
    test_preds_lr += lr_clf.predict_proba(X_full_test) / n_folds
    
    # Fold Score Check
    fold_pred = np.argmax(oof_preds_lgb[val_idx], axis=1)
    fold_f1 = f1_score(y_val, fold_pred, average='macro')
    print(f"Fold {fold+1} LGBM F1: {fold_f1:.5f}")

In [ ]:
!zip -r download_output.zip /kaggle/working/


In [ ]:
from IPython.display import FileLink
FileLink(r'download_output.zip')
